In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
data_dir = Path('data')
train_df = pd.read_csv(data_dir / 'train.csv')
test_df = pd.read_csv(data_dir / 'test.csv')
train_df.head(1)

OSError: [Errno 22] Invalid argument: '\\data\train.csv'

In [ ]:
all_features = train_df.columns.tolist()
print(all_features)

In [ ]:
test_df['Total_Area'] = test_df['GrLivArea'] + test_df['TotalBsmtSF']
test_df['Total_Baths'] = test_df['FullBath'] + (0.5 * test_df['HalfBath']) + test_df['BsmtFullBath'] + (0.5 * test_df['BsmtHalfBath'])

In [ ]:
train_df['Total_Area'] = train_df['GrLivArea'] + train_df['TotalBsmtSF']
train_df['Total_Baths'] = train_df['FullBath'] + (0.5 * train_df['HalfBath']) + train_df['BsmtFullBath'] + (0.5 * train_df['BsmtHalfBath'])

In [ ]:
features = ['BedroomAbvGr', 'Total_Area' , 'Total_Baths']
target = 'SalePrice'

In [ ]:
X_train_full = train_df[features].fillna(train_df[features].median())
y_train_full = train_df[target]

X_test_final = test_df[features].fillna(train_df[features].median())

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)

In [ ]:
# Create a list of the features
features = ['BedroomAbvGr', 'Total_Area', 'Total_Baths']

# Fill any empty cells with the median value of the training data
for col in features:
    median_value = train_df[col].median()
    train_df[col] = train_df[col].fillna(median_value)
    test_df[col] = test_df[col].fillna(median_value)

print("Missing values successfully filled!")

In [ ]:
# Initialize the model
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate the model performance on our validation set
y_val_pred = model.fit(X_train, y_train).predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
r2 = r2_score(y_val, y_val_pred)

print(f"Validation Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"Validation R² Score: {r2:.4f}")


In [ ]:
# Plot Actual vs Predicted Prices
plt.figure(figsize=(8, 6))
plt.scatter(y_val, y_val_pred, alpha=0.5, color='blue')
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], color='red', lw=2, linestyle='--')
plt.title('Actual vs. Predicted House Prices (Validation Set)')
plt.xlabel('Actual SalePrice ($)')
plt.ylabel('Predicted SalePrice ($)')
plt.grid(True)
plt.show()

In [ ]:
# Generate predictions on the actual Kaggle test dataset
test_predictions = model.predict(X_test_final)

In [ ]:
# Generate predictions using the cleaned test data features
test_predictions = model.predict(test_df[features])

# Combine the House IDs with their newly predicted prices
results_df = pd.DataFrame({
    'House_Id': test_df['Id'],
    'Predicted_SalePrice': test_predictions
})

# Display the first 15 predictions to see what the model guessed
print("--- Model Guesses for Unseen Test Data ---")
print(results_df.head(15))